In [ ]:
import pandas as pd
import json
import os
from google.colab import drive
from google.colab import files

# ==========================================
# 1. SETUP & FILE LOADING
# ==========================================

# Mount Google Drive
drive.mount('/content/drive')

# Define file names
filename = '/content/drive/MyDrive/datasets/Assertive sentence - Sheet1.csv'
output_filename = 'nllb_multilingual_assertive_dataset.json'

# Paths to check
possible_paths = [
    f'/content/{filename}',                        # Direct upload
    f'/content/drive/My Drive/{filename}',         # Root of Drive
    f'/content/drive/MyDrive/{filename}',          # Root of Drive (alternate)
]

input_path = None

# Check if file exists in Drive or local storage
for path in possible_paths:
    if os.path.exists(path):
        input_path = path
        print(f"✅ Found file at: {input_path}")
        break

# If not found, ask user to upload it NOW
if input_path is None:
    print(f"⚠️ File '{filename}' not found automatically.")
    print("Please upload the CSV file now using the button below:")
    uploaded = files.upload()
    input_path = list(uploaded.keys())[0] # Get the name of the uploaded file
    print(f"✅ File uploaded: {input_path}")

# ==========================================
# 2. DATA CLEANING FUNCTIONS
# ==========================================

def clean_text(text, script_type):
    """
    Cleans text by stripping whitespace and ensuring correct punctuation.
    script_type: 'latn' (English/Marma-Eng) -> adds '.'
    script_type: 'beng' (Bangla/Marma-Ben)  -> adds '।'
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()
    if not text:
        return ""

    # Danda character for Bangla
    danda = '।'

    if script_type == 'latn':
        # If it doesn't end with . ! or ?, add .
        if text[-1] not in ['.', '!', '?']:
            text += '.'
    elif script_type == 'beng':
        # If it doesn't end with | ! ? or ., add |
        if text[-1] not in [danda, '!', '?', '.']:
            text += danda

    return text

# ==========================================
# 3. PROCESSING & AUGMENTATION
# ==========================================

try:
    print("Reading CSV file...")
    df = pd.read_csv(input_path)

    # Validate columns exist
    required_cols = ['English', 'Bangla', 'Marma_English_Transliterate', 'Marma_Bangla_Transliterate']
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"CSV is missing columns. Found: {df.columns}. Needed: {required_cols}")

    print("Cleaning punctuation...")
    # Apply cleaning based on script type
    df['English'] = df['English'].apply(lambda x: clean_text(x, 'latn'))
    df['Marma_English_Transliterate'] = df['Marma_English_Transliterate'].apply(lambda x: clean_text(x, 'latn'))
    df['Bangla'] = df['Bangla'].apply(lambda x: clean_text(x, 'beng'))
    df['Marma_Bangla_Transliterate'] = df['Marma_Bangla_Transliterate'].apply(lambda x: clean_text(x, 'beng'))

    print("Augmenting data (Generating All Pairs)...")
    augmented_dataset = []

    # Iterate through rows and generate permutations
    for index, row in df.iterrows():
        # Map text to language codes
        texts = {
            'eng_Latn': row['English'],
            'ben_Beng': row['Bangla'],
            'mrm_Latn': row['Marma_English_Transliterate'],
            'mrm_Beng': row['Marma_Bangla_Transliterate']
        }

        # Generate every possible pair (Source -> Target)
        keys = list(texts.keys())
        for src_lang in keys:
            for tgt_lang in keys:
                if src_lang != tgt_lang:
                    # Skip if either source or target is empty
                    if not texts[src_lang] or not texts[tgt_lang]:
                        continue

                    # NLLB / Unsloth JSON Structure
                    sample = {
                        "translation": {
                            "src": texts[src_lang],
                            "tgt": texts[tgt_lang],
                            "src_lang": src_lang,
                            "tgt_lang": tgt_lang
                        }
                    }
                    augmented_dataset.append(sample)

    # ==========================================
    # 4. SAVING TO DRIVE
    # ==========================================

    # Define save path in Drive
    save_path = f'/content/drive/My Drive/{output_filename}'

    print(f"Saving {len(augmented_dataset)} translation pairs to Google Drive...")
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(augmented_dataset, f, ensure_ascii=False, indent=2)

    print("✅ SUCCESS!")
    print(f"Dataset saved correctly at: {save_path}")
    print("Check your Google Drive folder for 'nllb_multilingual_assertive_dataset.json'")

except Exception as e:
    print(f"\n❌ An error occurred during processing: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️ File '/content/drive/MyDrive/datasets/Assertive sentence - Sheet1.csv' not found automatically.
Please upload the CSV file now using the button below:


Saving Assertive sentence - Sheet1 (1).csv to Assertive sentence - Sheet1 (1).csv
✅ File uploaded: Assertive sentence - Sheet1 (1).csv
Reading CSV file...
Cleaning punctuation...
Augmenting data (Generating All Pairs)...
Saving 65138 translation pairs to Google Drive...
✅ SUCCESS!
Dataset saved correctly at: /content/drive/My Drive/nllb_multilingual_assertive_dataset.json
Check your Google Drive folder for 'nllb_multilingual_assertive_dataset.json'
